# 03 - Cross-generation (GPT-3.5-turbo) and frontier validation (GPT-5.5)

Both evaluated on the identical 360-sample benchmark.
Run the GPT-5.5 pilot cell first to measure real cost per call.

In [1]:
# --- environment ---
!pip -q install openai
from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')   # SARD corpus root
WORK = ROOT / 'revision_2026'                              # outputs live here
WORK.mkdir(exist_ok=True)

sys.path.insert(0, str(WORK))          # vulnbench.py lives in WORK
import vulnbench as vb

BENCH = pd.read_csv(WORK / 'benchmark_360_metadata.csv')
print(len(BENCH), 'samples |', BENCH.true_label.value_counts().to_dict())

Mounted at /content/drive
360 samples | {'Safe': 180, 'Vulnerable': 180}


In [2]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
from openai import OpenAI
client = OpenAI()

OpenAI API key: ··········


In [4]:
# ---- E_full 成本探針：12 次呼叫 ----
probeE = BENCH.groupby('true_label', group_keys=False).sample(6, random_state=7)

rE = vb.run_experiment(
    bench        = probeE,
    dataset_root = ROOT,
    conditions   = [('E', 'clean')],
    model        = 'gpt-5.5-2026-04-23',
    temperature  = None,
    out_csv      = WORK / 'probe_gpt55_E.csv',
    client       = client,
)
print(pd.crosstab(rE.true_label, rE.prediction))

12 calls to make with gpt-5.5-2026-04-23
  12/12  |  tokens in/out 3897/4993  |  running cost $0.169

done — 12 rows in /content/drive/MyDrive/LLM_Security_Paper/revision_2026/probe_gpt55_E.csv
this session cost ≈ $0.169
prediction  Safe  Vulnerable
true_label                  
Safe           4           2
Vulnerable     2           4


In [6]:
# --- GPT-3.5-turbo on the IDENTICAL 360 samples, all 10 conditions ---
res35 = vb.run_experiment(
    bench        = BENCH,
    dataset_root = ROOT,
    conditions   = [(v, h) for v in 'ABCDE' for h in ('clean', 'hinted')],
    model        = 'gpt-3.5-turbo-0125',
    temperature  = 0.1,
    out_csv      = WORK / 'crossgen_gpt35.csv',
    client       = client,
)

3600 calls to make with gpt-3.5-turbo-0125
  50/3600  |  tokens in/out 9135/1450  |  running cost $0.007
  100/3600  |  tokens in/out 18770/2900  |  running cost $0.014
  150/3600  |  tokens in/out 31859/4350  |  running cost $0.022
  200/3600  |  tokens in/out 46272/5800  |  running cost $0.032
  250/3600  |  tokens in/out 60144/7250  |  running cost $0.041
  300/3600  |  tokens in/out 69132/8700  |  running cost $0.048
  350/3600  |  tokens in/out 79134/10150  |  running cost $0.055
  400/3600  |  tokens in/out 89040/11596  |  running cost $0.062
  450/3600  |  tokens in/out 99293/13042  |  running cost $0.069
  500/3600  |  tokens in/out 111997/14480  |  running cost $0.078
  550/3600  |  tokens in/out 127522/15930  |  running cost $0.088
  600/3600  |  tokens in/out 143232/17380  |  running cost $0.098
  650/3600  |  tokens in/out 153062/18830  |  running cost $0.105
  700/3600  |  tokens in/out 163294/20276  |  running cost $0.112
  750/3600  |  tokens in/out 173391/21726  |  runn

In [3]:
# ---- GPT-5.5 洩漏探針：12 Safe + 12 Vulnerable，24 次呼叫約 $0.36 ----
probe5 = BENCH.groupby('true_label', group_keys=False).sample(12, random_state=1)
print(len(probe5), probe5.true_label.value_counts().to_dict())

r5 = vb.run_experiment(
    bench        = probe5,
    dataset_root = ROOT,
    conditions   = [('A', 'clean')],
    model        = 'gpt-5.5-2026-04-23',
    temperature  = None,
    out_csv      = WORK / 'probe_gpt55.csv',
    client       = client,
)

print(pd.crosstab(r5.true_label, r5.prediction))
print('\nSafe 判對:', ((r5.true_label=='Safe') & (r5.prediction=='Safe')).sum(), '/ 12')
print('Vuln 判對:', ((r5.true_label=='Vulnerable') & (r5.prediction=='Vulnerable')).sum(), '/ 12')

24 {'Safe': 12, 'Vulnerable': 12}
24 calls to make with gpt-5.5-2026-04-23
  24/24  |  tokens in/out 6015/11589  |  running cost $0.378

done — 24 rows in /content/drive/MyDrive/LLM_Security_Paper/revision_2026/probe_gpt55.csv
this session cost ≈ $0.378
prediction  Safe  Vulnerable
true_label                  
Safe           7           5
Vulnerable     1          11

Safe 判對: 7 / 12
Vuln 判對: 11 / 12


In [5]:
# --- GPT-5.5 full run: Baseline vs Full Framework, CLEAN form ---
res55 = vb.run_experiment(
    bench        = BENCH,
    dataset_root = ROOT,
    conditions   = [('A', 'clean'), ('E', 'clean')],
    model        = 'gpt-5.5-2026-04-23',
    temperature  = None,
    out_csv      = WORK / 'frontier_gpt55.csv',
    client       = client,
)

720 calls to make with gpt-5.5-2026-04-23
  50/720  |  tokens in/out 9241/18600  |  running cost $0.604
  100/720  |  tokens in/out 18968/48559  |  running cost $1.552
  150/720  |  tokens in/out 32179/72086  |  running cost $2.323
  200/720  |  tokens in/out 46710/92288  |  running cost $3.002
  250/720  |  tokens in/out 60695/115733  |  running cost $3.775
  300/720  |  tokens in/out 69799/139649  |  running cost $4.538
  350/720  |  tokens in/out 79922/166641  |  running cost $5.399
  400/720  |  tokens in/out 93463/185751  |  running cost $6.040
  450/720  |  tokens in/out 108206/215707  |  running cost $7.012
  500/720  |  tokens in/out 125452/243499  |  running cost $7.932
  550/720  |  tokens in/out 145551/260800  |  running cost $8.552
  600/720  |  tokens in/out 165812/285207  |  running cost $9.385
  650/720  |  tokens in/out 180118/307932  |  running cost $10.139
  700/720  |  tokens in/out 194811/329163  |  running cost $10.849
  720/720  |  tokens in/out 201266/338924  |  

In [ ]:
p = pd.read_csv(WORK / 'frontier_pilot.csv')
print(p[['sample_id','sanitizer','true_label','prediction','confidence']].to_string(index=False))
print('\n判對:', (p.prediction == p.true_label).sum(), '/', len(p))

sample_id                                sanitizer true_label prediction  confidence
    S0001 func_FILTER-CLEANING-magic_quotes_filter       Safe Vulnerable          95
    S0002                        func_preg_replace       Safe       Safe          90
    S0003                       ternary_white_list       Safe       Safe          95
    S0004                    whitelist_using_array       Safe       Safe          98
    S0005                  CAST-cast_float_sort_of       Safe       Safe          95
    S0006                          CAST-cast_float       Safe       Safe          95
    S0007                          CAST-cast_float       Safe       Safe          95
    S0008                          func_addslashes       Safe Vulnerable          95
    S0009                       ternary_white_list       Safe       Safe          95
    S0010                            CAST-cast_int       Safe       Safe          95

判對: 8 / 10
